# Bronze Layer — Raw Transaction Ingestion

**Objective:** Ingest fictional fintech transaction data from CSV files and persist the raw records in a managed Delta Lake table using Databricks and PySpark.

### Why?

**For our project:** Preserves the original transaction records, including duplicates and invalid values, for subsequent validation in the Silver layer.

**For Data Engineering:** Demonstrates reliable raw-data ingestion, Delta Lake persistence, and separation of responsibilities within the Medallion architecture.

#### 01 — Read Source Transactions

Read the initial CSV batch as strings to preserve the original values. This allows data types and invalid records to be handled explicitly in the Silver layer.

In [0]:
# Preserve raw field types; type conversion occurs in Silver
df_bronze_raw = spark.read.csv("/Volumes/workspace/default/fintech_landing/transactions_batch_01.csv",
    header=True,
    inferSchema=False
)

display(df_bronze_raw)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


#### 02 — Inspect Source Schema

Verify that the six source columns have STRING data types. This confirms that the raw schema is preserved before transformation.

In [0]:
df_bronze_raw.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)



#### 03 — Validate Source Record Count

Count the source records and verify that Batch 01 contains exactly 12 transactions. The assertion detects missing or unexpected records before data persistence.

In [0]:
df_bronze_raw.count()

12

In [0]:
# Stop execution if the source record count is incorrect
assert df_bronze_raw.count() == 12, "Unexpected Bronze row count"

#### 04 — Persist Bronze Delta Table

Create a managed Delta Lake table to persist the original transactions, including duplicates and invalid records. This preserves the raw data for subsequent processing and traceability.

The `error` write mode prevents accidental replacement of an existing table.

In [0]:
# Initial table creation; fail if the table already exists
df_bronze_raw.write.format("delta").mode("error").saveAsTable("workspace.default.fintech_transactions_bronze")

#### 05 — Inspect Persisted Bronze Records

Read the persisted Delta table and inspect its records. This confirms that the raw transaction data is accessible for downstream processing.

In [0]:
df_bronze_saved = spark.table(
    "workspace.default.fintech_transactions_bronze"
)

display(df_bronze_saved)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX001,C001,2026-10-18 09:00:00,PIX_IN,250.00,COMPLETED
TX002,C001,2026-10-18 09:30:00,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18 10:00:00,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18 11:00:00,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18 12:00:00,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18 13:00:00,PIX_OUT,120.00,FAILED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX007,C005,2026-10-18 13:20:00,PIX_IN,80.00,COMPLETED
TX008,C006,2026-10-18 14:00:00,PIX_OUT,35.00,COMPLETED
null,C007,2026-10-18 14:15:00,CARD_PURCHASE,25.00,COMPLETED


#### 06 — Validate Persisted Record Count

Verify that the initial Bronze table contains all 12 Batch 01 transactions. This helps detect missing or additional records after persistence.

In [0]:
# Validate the initial Batch 01 table state
assert df_bronze_saved.count() == 12, "Bronze row count mismatch"

#### 07 — Inspect Delta Transaction History

Inspect the Delta transaction history to verify the initial table creation. Delta history provides metadata about table operations, supporting traceability and auditing.

In [0]:
%sql
DESCRIBE HISTORY workspace.default.fintech_transactions_bronze;

#### 08 — Validate Persisted Column Count

Verify that the persisted Bronze table contains six columns. This provides a basic structural check before downstream transformations.

In [0]:
assert len(df_bronze_saved.columns) == 6,    "Unexpected number of columns"